## **0. Library import**

In [1]:
import os
import sys

# Add the root path into the python path
root_path = os.path.abspath(os.path.join(".."))
if not root_path in sys.path:
    sys.path.insert(0, root_path)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.config import BRFSS_FILTERING_FILE_PATH, BALANCING_STRATEGY, TRAIN_SIZE
from src.features import DiabetesFeatureEngineering
from src.balancing.over_sampler import OverSamplingBalancer
from src.balancing.under_sampler import UnderSamplingBalancer
from src.balancing.hyprid import HybridSamplingBalancer

## **1. Load data**

In [3]:
df = pd.read_csv(BRFSS_FILTERING_FILE_PATH)
df.head()

,Diabetes,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Year
0,2.0,1.0,1.0,1.0,27.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,1.0,0.0,11.0,6.0,6.0,2017
1,0.0,1.0,0.0,1.0,29.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,1.0,10.0,6.0,8.0,2017
2,0.0,0.0,0.0,1.0,23.0,1.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,14.0,0.0,0.0,10.0,2.0,2.0,2017
3,0.0,1.0,0.0,1.0,27.0,1.0,0.0,1.0,1.0,0.0,...,0.0,3.0,0.0,6.0,0.0,1.0,12.0,4.0,4.0,2017
4,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,0.0,0.0,...,0.0,3.0,0.0,0.0,0.0,1.0,10.0,5.0,8.0,2017


## **2. Feature engineering**

In [4]:
diabetes_feature_engineering = DiabetesFeatureEngineering("../logs/data_balancing.log")
processed_df, _, _ = diabetes_feature_engineering.process_all(df)
processed_df.shape

2025-07-20 10:27:02,106 - [src.features] - INFO - DiabetesFeatureEngineering initialized successfully
2025-07-20 10:27:02,107 - [src.features] - INFO - ============================================================
2025-07-20 10:27:02,107 - [src.features] - INFO - STARTING COMPLETE FEATURE ENGINEERING PIPELINE
2025-07-20 10:27:02,108 - [src.features] - INFO - ============================================================
2025-07-20 10:27:02,108 - [src.features] - INFO - Initial dataset shape: (787646, 21)
2025-07-20 10:27:02,108 - [src.features] - INFO - Starting null values removal process...
2025-07-20 10:27:02,124 - [src.features] - INFO - No null values found in the dataset
2025-07-20 10:27:02,300 - [src.features] - INFO - Null values removal completed. Removed 0 rows (0.00%)
2025-07-20 10:27:02,301 - [src.features] - INFO - Dataset shape: 787646 -> 787646 rows
2025-07-20 10:27:02,301 - [src.features] - INFO - After null removal: (787646, 21)
2025-07-20 10:27:02,302 - [src.features] - 

(702560, 15)

In [5]:
processed_df["Diabetes"].value_counts()

Diabetes
0.0    574477
2.0    111193
1.0     16890
Name: count, dtype: int64

In [6]:
X = processed_df.drop(columns=["Diabetes"])
y = processed_df["Diabetes"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=TRAIN_SIZE, random_state=42, stratify=y)

## **3. Data balancing**

### **3.1 Over sampling**

In [8]:
over_sampling_balancer = OverSamplingBalancer()

2025-07-20 10:27:07,748 - [src.balancing.over_sampler] - INFO - OverSamplingBalancer initialized successfully


#### **3.1.1 Random Over Sampling**

In [9]:
ros_X_train, ros_y_train = over_sampling_balancer.apply_random_oversampling(X_train, y_train, BALANCING_STRATEGY)

2025-07-20 10:27:07,796 - [src.balancing.over_sampler] - INFO - Starting Random Over Sampling process...
2025-07-20 10:27:07,802 - [src.balancing.over_sampler] - INFO - Original class distribution:
2025-07-20 10:27:07,802 - [src.balancing.over_sampler] - INFO -   - Class 0.0: 459582 samples
2025-07-20 10:27:07,803 - [src.balancing.over_sampler] - INFO -   - Class 1.0: 13512 samples
2025-07-20 10:27:07,803 - [src.balancing.over_sampler] - INFO -   - Class 2.0: 88954 samples
2025-07-20 10:27:08,090 - [src.balancing.over_sampler] - INFO - New class distribution after Random Over Sampling:
2025-07-20 10:27:08,091 - [src.balancing.over_sampler] - INFO -   - Class 0.0: 574477 samples (+114895)
2025-07-20 10:27:08,091 - [src.balancing.over_sampler] - INFO -   - Class 1.0: 150000 samples (+136488)
2025-07-20 10:27:08,092 - [src.balancing.over_sampler] - INFO -   - Class 2.0: 100000 samples (+11046)
2025-07-20 10:27:08,092 - [src.balancing.over_sampler] - INFO - Dataset size: 562048 -> 824477 s

In [24]:
ros_y_train.value_counts()

Diabetes
0.0    574477
1.0    150000
2.0    100000
Name: count, dtype: int64

#### **3.1.2 SMOTE**

In [11]:
smote_X_train, smote_y_train = over_sampling_balancer.apply_smote(X_train, y_train, BALANCING_STRATEGY)

2025-07-20 10:27:08,257 - [src.balancing.over_sampler] - INFO - Starting SMOTE process...
2025-07-20 10:27:08,262 - [src.balancing.over_sampler] - INFO - Original class distribution:
2025-07-20 10:27:08,262 - [src.balancing.over_sampler] - INFO -   - Class 0.0: 459582 samples
2025-07-20 10:27:08,263 - [src.balancing.over_sampler] - INFO -   - Class 1.0: 13512 samples
2025-07-20 10:27:08,263 - [src.balancing.over_sampler] - INFO -   - Class 2.0: 88954 samples
2025-07-20 10:28:33,352 - [src.balancing.over_sampler] - INFO - New class distribution after SMOTE:
2025-07-20 10:28:33,353 - [src.balancing.over_sampler] - INFO -   - Class 0.0: 574477 samples (+114895 synthetic)
2025-07-20 10:28:33,353 - [src.balancing.over_sampler] - INFO -   - Class 1.0: 150000 samples (+136488 synthetic)
2025-07-20 10:28:33,354 - [src.balancing.over_sampler] - INFO -   - Class 2.0: 100000 samples (+11046 synthetic)
2025-07-20 10:28:33,354 - [src.balancing.over_sampler] - INFO - Dataset size: 562048 -> 824477 s

In [12]:
smote_y_train.value_counts()

Diabetes
0.0    574477
1.0    150000
2.0    100000
Name: count, dtype: int64

### **3.2 Under Sampling**

In [13]:
under_sampling_balancer = UnderSamplingBalancer()

2025-07-20 10:28:33,517 - [src.balancing.under_sampler] - INFO - UnderSamplingBalancer initialized successfully


#### **3.2.1 Random Under Sampling**

In [14]:
rus_X_train, rus_y_train = under_sampling_balancer.apply_random_undersampling(X_train, y_train)

2025-07-20 10:28:33,622 - [src.balancing.under_sampler] - INFO - Starting Random Under Sampling process...
2025-07-20 10:28:33,631 - [src.balancing.under_sampler] - INFO - Original class distribution:
2025-07-20 10:28:33,632 - [src.balancing.under_sampler] - INFO -   - Class 0.0: 459582 samples
2025-07-20 10:28:33,633 - [src.balancing.under_sampler] - INFO -   - Class 1.0: 13512 samples
2025-07-20 10:28:33,634 - [src.balancing.under_sampler] - INFO -   - Class 2.0: 88954 samples
2025-07-20 10:28:33,727 - [src.balancing.under_sampler] - INFO - New class distribution after Random Under Sampling:
2025-07-20 10:28:33,728 - [src.balancing.under_sampler] - INFO -   - Class 0.0: 13512 samples (-446070)
2025-07-20 10:28:33,729 - [src.balancing.under_sampler] - INFO -   - Class 1.0: 13512 samples (-0)
2025-07-20 10:28:33,730 - [src.balancing.under_sampler] - INFO -   - Class 2.0: 13512 samples (-75442)
2025-07-20 10:28:33,731 - [src.balancing.under_sampler] - INFO - Dataset size: 562048 -> 4053

In [15]:
rus_y_train.value_counts()

Diabetes
0.0    13512
1.0    13512
2.0    13512
Name: count, dtype: int64

#### **3.2.2 TomekLinks**

In [16]:
tomek_X_train, tomek_y_train = under_sampling_balancer.apply_tomek_links(X_train, y_train)

2025-07-20 10:28:33,953 - [src.balancing.under_sampler] - INFO - Starting Tomek Links process...
2025-07-20 10:28:33,960 - [src.balancing.under_sampler] - INFO - Original class distribution:
2025-07-20 10:28:33,961 - [src.balancing.under_sampler] - INFO -   - Class 0.0: 459582 samples
2025-07-20 10:28:33,961 - [src.balancing.under_sampler] - INFO -   - Class 1.0: 13512 samples
2025-07-20 10:28:33,962 - [src.balancing.under_sampler] - INFO -   - Class 2.0: 88954 samples
2025-07-20 10:29:32,210 - [src.balancing.under_sampler] - INFO - New class distribution after Tomek Links:
2025-07-20 10:29:32,211 - [src.balancing.under_sampler] - INFO -   - Class 0.0: 444220 samples (-15362 Tomek links)
2025-07-20 10:29:32,211 - [src.balancing.under_sampler] - INFO -   - Class 1.0: 13512 samples (unchanged)
2025-07-20 10:29:32,211 - [src.balancing.under_sampler] - INFO -   - Class 2.0: 74822 samples (-14132 Tomek links)
2025-07-20 10:29:32,212 - [src.balancing.under_sampler] - INFO - Dataset size: 562

In [17]:
tomek_y_train.value_counts()

Diabetes
0.0    444220
2.0     74822
1.0     13512
Name: count, dtype: int64

### **3.3 Hyprid**

In [18]:
hyprid_sampler = HybridSamplingBalancer()

2025-07-20 10:29:32,402 - [src.balancing.hyprid] - INFO - HybridSamplingBalancer initialized successfully


#### **3.3.1 SMOTETomek**

In [19]:
smote_tomek_X_train, smote_tomek_y_train = hyprid_sampler.apply_smote_tomek(X_train, y_train, BALANCING_STRATEGY, os.cpu_count())

2025-07-20 10:29:32,489 - [src.balancing.hyprid] - INFO - Starting SMOTETomek hybrid sampling process...
2025-07-20 10:29:32,495 - [src.balancing.hyprid] - INFO - Original class distribution:
2025-07-20 10:29:32,495 - [src.balancing.hyprid] - INFO -   - Class 0.0: 459582 samples
2025-07-20 10:29:32,496 - [src.balancing.hyprid] - INFO -   - Class 1.0: 13512 samples
2025-07-20 10:29:32,496 - [src.balancing.hyprid] - INFO -   - Class 2.0: 88954 samples
2025-07-20 10:31:39,981 - [src.balancing.hyprid] - INFO - New class distribution after SMOTETomek:
2025-07-20 10:31:39,982 - [src.balancing.hyprid] - INFO -   - Class 0.0: 564281 samples (+104699 net)
2025-07-20 10:31:39,982 - [src.balancing.hyprid] - INFO -   - Class 1.0: 149023 samples (+135511 net)
2025-07-20 10:31:39,982 - [src.balancing.hyprid] - INFO -   - Class 2.0: 90337 samples (+1383 net)
2025-07-20 10:31:39,983 - [src.balancing.hyprid] - INFO - Dataset size: 562048 -> 803641 samples (+241593)
2025-07-20 10:31:39,983 - [src.balanc

In [23]:
smote_tomek_y_train.value_counts()

Diabetes
0.0    564281
1.0    149023
2.0     90337
Name: count, dtype: int64

#### **3.3.2 SMOTEENN**

In [21]:
smoteen_X_train, smoteen_y_train = hyprid_sampler.apply_smote_enn(X_train, y_train, BALANCING_STRATEGY, os.cpu_count())

2025-07-20 10:31:40,188 - [src.balancing.hyprid] - INFO - Starting SMOTEENN hybrid sampling process...
2025-07-20 10:31:40,194 - [src.balancing.hyprid] - INFO - Original class distribution:
2025-07-20 10:31:40,195 - [src.balancing.hyprid] - INFO -   - Class 0.0: 459582 samples
2025-07-20 10:31:40,195 - [src.balancing.hyprid] - INFO -   - Class 1.0: 13512 samples
2025-07-20 10:31:40,195 - [src.balancing.hyprid] - INFO -   - Class 2.0: 88954 samples
2025-07-20 10:34:19,752 - [src.balancing.hyprid] - INFO - New class distribution after SMOTEENN:
2025-07-20 10:34:19,754 - [src.balancing.hyprid] - INFO -   - Class 0.0: 405247 samples (-54335 net)
2025-07-20 10:34:19,754 - [src.balancing.hyprid] - INFO -   - Class 1.0: 105910 samples (+92398 net)
2025-07-20 10:34:19,754 - [src.balancing.hyprid] - INFO -   - Class 2.0: 8814 samples (-80140 net)
2025-07-20 10:34:19,755 - [src.balancing.hyprid] - INFO - Dataset size: 562048 -> 519971 samples (-42077)
2025-07-20 10:34:19,755 - [src.balancing.hyp

In [22]:
smoteen_y_train.value_counts()

Diabetes
0.0    405247
1.0    105910
2.0      8814
Name: count, dtype: int64